# RetailIQ - Phase 5a: Sales Forecasting

## Overview
This notebook covers **Mission 5a (Part 1)**: training and evaluating an XGBoost Regressor to forecast daily total revenue. The model leverages the engineered calendar, lag, and rolling-average features from Phase 4.

## 1. Environment Setup & Data Loading
We import standard packages, load our forecasting dataset, and clean missing values resulting from the lag features.

In [1]:
import os
import pickle
import numpy as np
import pandas as pd
from xgboost import XGBRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, mean_absolute_percentage_error

DATA_PATH = os.path.join("..", "data", "processed", "forecast_dataset.csv")
df = pd.read_csv(DATA_PATH)
df['Date'] = pd.to_datetime(df['Date'])
df_clean = df.dropna().copy()
df_clean.sort_values('Date', inplace=True)
print("Setup complete. Data loaded.")

Setup complete. Data loaded.


## 2. Time-Based Train/Test Split
To ensure a realistic evaluation for time-series forecasting, we split the data sequentially without random shuffling. The last 8 weeks (56 days) represent the test set.

In [2]:
test_size = 56
train_df = df_clean.iloc[:-test_size]
test_df = df_clean.iloc[-test_size:]

features = [
    'DayOfWeek', 'Month', 'Quarter', 'Year', 'IsWeekend', 'IsHolidaySeason',
    'Revenue_Lag_7', 'Volume_Lag_7', 'Revenue_Lag_30', 'Volume_Lag_30',
    'Revenue_RollMean_7', 'Volume_RollMean_7', 'Revenue_RollMean_30', 'Volume_RollMean_30'
]
target = 'TotalRevenue'

X_train, y_train = train_df[features], train_df[target]
X_test, y_test = test_df[features], test_df[target]
print(f"Train set size: {len(train_df)} days ({train_df['Date'].min().strftime('%Y-%m-%d')} to {train_df['Date'].max().strftime('%Y-%m-%d')})")
print(f"Test set size: {len(test_df)} days ({test_df['Date'].min().strftime('%Y-%m-%d')} to {test_df['Date'].max().strftime('%Y-%m-%d')})")

Train set size: 645 days (2023-10-10 to 2025-07-15)
Test set size: 56 days (2025-07-16 to 2025-09-09)


## 3. Train XGBoost Regressor
We train a single primary model (XGBoost Regressor) on the features.

> **Note on Prophet / LSTM:** Other model choices like Facebook Prophet or LSTM neural networks are documented here as future extensions for multivariate time-series. We select XGBoost due to its robust handling of lag/rolling features and tabular calendar variables.

In [3]:
xgb_model = XGBRegressor(n_estimators=100, max_depth=5, learning_rate=0.05, random_state=42)
xgb_model.fit(X_train, y_train)
print("Model trained successfully.")

Model trained successfully.


## 4. Evaluation and Visualization
We evaluate the model predictions over the test period using RMSE, MAE, and MAPE (excluding zero-revenue days to prevent division by zero errors).

In [4]:
y_pred = xgb_model.predict(X_test)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
mae = mean_absolute_error(y_test, y_pred)
valid_mask = y_test > 0
mape = mean_absolute_percentage_error(y_test[valid_mask], y_pred[valid_mask]) * 100

print("Evaluation Metrics:")
print(f"RMSE: {rmse:.2f}")
print(f"MAE: {mae:.2f}")
print(f"MAPE (excl. 0s): {mape:.2f}%")

# Plotting logic
# (Chart pre-rendered from reports/sales_forecast_test.png)

Evaluation Metrics:
RMSE: 10267.82
MAE: 8117.92
MAPE (excl. 0s): 54.50%


## 5. Feature Importances
We extract the feature importances to identify which variables are most predictive of sales revenue.

In [5]:
# Plotting feature importances
# (Chart pre-rendered from reports/sales_forecast_importance.png)

#### Business Interpretation:
Short-term rolling features (`Revenue_RollMean_7`) and historical volumes (`Volume_RollMean_30`, `Volume_Lag_30`) strongly dominate the feature importance metrics. This indicates that sales trends and short-term operational momentum are much more predictive of next-day sales than general calendar indicators like month or day-of-week.

## 6. Model Serialization & Predictions Export
We serialize the trained estimator and save the test actuals vs. predictions to a CSV file to feed downstream Power BI dashboards.

In [6]:
MODELS_DIR = os.path.join("..", "models")
PROCESSED_DIR = os.path.join("..", "data", "processed")
os.makedirs(MODELS_DIR, exist_ok=True)

with open(os.path.join(MODELS_DIR, "sales_forecast_model.pkl"), "wb") as f:
    pickle.dump(xgb_model, f)
print("Saved model to models/sales_forecast_model.pkl")

pred_df = pd.DataFrame({
    'Date': test_df['Date'],
    'ActualRevenue': y_test,
    'PredictedRevenue': y_pred
})
pred_df.to_csv(os.path.join(PROCESSED_DIR, "forecast_predictions.csv"), index=False)
print("Saved predictions to data/processed/forecast_predictions.csv")

Saved model to models/sales_forecast_model.pkl
Saved predictions to data/processed/forecast_predictions.csv
